# Midpoint past-tense run: regular → irregular transfer by verb class

Copy of `notebooks/analogy/run.ipynb` for the Milestone 5 run. Changes: a `classes_path` and `num_samples` parameter; the automatic verb class of each VBD pair attached to `all_cross_instances`; the 20 past-tense experiments in place of the defaults; the generated experiments removed; partial results named by experiment. Run it with `scripts/past_tense/02_run_analogies.py`.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import itertools
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import seaborn as sns
from tqdm import tqdm

from src.analysis import analogy
from src.analysis.state_space import StateSpaceAnalysisSpec, \
    prepare_state_trajectory, flatten_trajectory
from src.datasets.speech_equivalence import SpeechHiddenStateDataset


In [ ]:
base_model = "w2v2_8"

model_class = "discrim-rnn_32-mAP1"
model_name = "word_broad_10frames_fixedlen25"

inflection_results_path = "inflection_results.parquet"
# all_cross_instances_path = "all_cross_instances.parquet"
all_cross_instances_path = "outputs/analogy/inputs/librispeech-train-clean-100/w2v2/all_cross_instances.parquet"
most_common_allomorphs_path = "most_common_allomorphs.csv"
false_friends_path = "false_friends.csv"

train_dataset = "librispeech-train-clean-100"
hidden_states_path = f"outputs/hidden_states/{base_model}/{train_dataset}.h5"
state_space_specs_path = f"state_space_spec.h5"
embeddings_path = f"outputs/model_embeddings/{train_dataset}/{base_model}/{model_class}/{model_name}/{train_dataset}.npy"

output_dir = f"."

pos_counts_path = "data/pos_counts.pkl"

seed = 42

metric = "cosine"

agg_fns = [
    "mean",
]

# midpoint past-tense run (scripts/past_tense/02_run_analogies.py)
classes_path = "outputs/past_tense/tables/vbd_classes.csv"
num_samples = 1000


In [ ]:
# Midpoint past-tense run: does the regular past-tense step transfer to irregular verbs, by verb class?
# Classes from scripts/past_tense/01_make_classes.py (automatic, not hand-checked); they are attached to
# all_cross_instances below (cell "attach verb classes"). Replaces the default experiments of
# notebooks/analogy/run.ipynb; the generated unambiguous / false-friend / forced-choice experiments are removed.
all_query = "not exclude_main"

R = "inflection == 'VBD' and past_class == 'regular'"
X = lambda c: f"inflection == 'VBD' and past_class == '{c}'"
RANDOM = "inflection == 'random'"
irregular_classes = ["semi_regular", "devoicing", "vowel_change", "suppletive"]

transfer = {"reg_to_reg": (R, R)}                                                     # reference
transfer |= {f"reg_to_{c}": (R, X(c)) for c in irregular_classes}                     # main
transfer |= {f"{c}_to_reg": (X(c), R) for c in ["semi_regular", "devoicing", "vowel_change"]}  # reverse
transfer |= {f"{c}_to_{c}": (X(c), X(c)) for c in ["semi_regular", "vowel_change", "devoicing"]}  # within class
transfer |= {f"random_to_{c}": (RANDOM, R if c == "reg" else X(c))                    # floor per target class
             for c in ["reg"] + irregular_classes}
transfer |= {f"reg_{a}_to_{c}": (f"{R} and allomorph == '{a}'", X(c))                 # [t]- vs [d]-source
             for c in ["semi_regular", "devoicing"] for a in ["T", "D"]}

experiments = {name: {"base_query": b, "inflected_query": i, "all_query": all_query}
               for name, (b, i) in transfer.items()}
assert len(experiments) == 20, len(experiments)
list(experiments)


In [ ]:
# TODO document
study_unambiguous_transfer = ["NNS", "VBZ"]
study_false_friends = ["NNS", "VBZ", "VBD"]

In [ ]:
fc_experiments = {
    ("Z", "S"): {
        "source_inflections": ["VBZ", "NNS"],
    },
    ("D", "T"): {
        "source_inflections": ["VBD"],
    },
    ("D", "IH D"): {
        "source_inflections": ["VBD"],
    },
    ("T", "IH D"): {
        "source_inflections": ["VBD"],
    },
}

## Prepare model representations

In [ ]:
if embeddings_path == "ID":
    model_representations = SpeechHiddenStateDataset.from_hdf5(hidden_states_path).states
else:
    with open(embeddings_path, "rb") as f:
        model_representations: np.ndarray = np.load(f)
state_space_spec = StateSpaceAnalysisSpec.from_hdf5(state_space_specs_path)
assert state_space_spec.is_compatible_with(model_representations)

In [ ]:
trajectory_agg = prepare_state_trajectory(model_representations, state_space_spec, 
                                          agg_fn_spec="mean", agg_fn_dimension=1)

In [ ]:
agg, agg_src = flatten_trajectory(trajectory_agg)
# the analogy computations run in float32; free the float64 copies (GBs for 768-dim states)
agg = agg.astype(np.float32)
del trajectory_agg, model_representations

## Prepare metadata

In [ ]:
cuts_df = state_space_spec.cuts.xs("phoneme", level="level").drop(columns=["onset_frame_idx", "offset_frame_idx"])
cuts_df["label_idx"] = cuts_df.index.get_level_values("label").map({l: i for i, l in enumerate(state_space_spec.labels)})
cuts_df["frame_idx"] = cuts_df.groupby(["label", "instance_idx"]).cumcount()
cuts_df = cuts_df.reset_index().set_index(["label", "instance_idx", "frame_idx"]).sort_index()

In [ ]:
cut_phonemic_forms = cuts_df.groupby(["label", "instance_idx"]).description.agg(' '.join)

In [ ]:
# Worldlex frequencies are not used in this notebook (only in the summary regression);
# load them if available.
from pathlib import Path
word_freq_path = Path("data/WorldLex_Eng_US.Freq.2.txt")
if word_freq_path.exists():
    word_freq_df = pd.read_csv(word_freq_path, sep="\t", index_col="Word")
    # compute weighted average frequency across domains
    word_freq_df["BlogFreq_rel"] = word_freq_df.BlogFreq / word_freq_df.BlogFreq.sum()
    word_freq_df["TwitterFreq_rel"] = word_freq_df.TwitterFreq / word_freq_df.TwitterFreq.sum()
    word_freq_df["NewsFreq_rel"] = word_freq_df.NewsFreq / word_freq_df.NewsFreq.sum()
    word_freq_df["Freq"] = word_freq_df[["BlogFreq", "TwitterFreq", "NewsFreq"]].mean(axis=1) \
        * word_freq_df[["BlogFreq", "TwitterFreq", "NewsFreq"]].sum().mean()
    word_freq_df["LogFreq"] = np.log10(word_freq_df.Freq)

In [ ]:
# Only load the columns used by the analogy experiments: the full table has ~15M rows on
# train-clean-100, and the unused phone-string columns dominate its memory footprint.
all_cross_instances_columns = [
    "inflection", "base", "inflected", "base_idx", "inflected_idx",
    "base_instance_idx", "inflected_instance_idx", "is_regular", "base_ambig_NN_VB",
    "post_divergence", "exclude_main",
]
all_cross_instances = pd.read_parquet(all_cross_instances_path, columns=all_cross_instances_columns)

# attach verb classes: the automatic class of each VBD pair (scripts/past_tense/01_make_classes.py).
# Assigned in place as categoricals, since a merge would copy this ~5 GB table; non-VBD rows get "none"
# so that string queries never meet NaN.
vbd_classes = pd.read_csv(classes_path, keep_default_na=False)
class_of = {(b, i): (c, a) for b, i, c, a in
            zip(vbd_classes.base, vbd_classes.inflected, vbd_classes.past_class, vbd_classes.allomorph)}
is_vbd = (all_cross_instances.inflection == "VBD").to_numpy()
vbd_keys = list(zip(all_cross_instances.base.to_numpy()[is_vbd], all_cross_instances.inflected.to_numpy()[is_vbd]))
missing = {k for k in set(vbd_keys) if k not in class_of}
assert not missing, f"VBD pairs without a class: {sorted(missing)[:10]}"
for col, j in [("past_class", 0), ("allomorph", 1)]:
    values = np.full(len(all_cross_instances), "none", dtype=object)
    values[is_vbd] = [class_of[k][j] for k in vbd_keys]
    all_cross_instances[col] = pd.Categorical(values)
del values, vbd_keys

vbd_main = all_cross_instances.loc[is_vbd & ~all_cross_instances.exclude_main.to_numpy(),
                                   ["base", "inflected", "past_class", "allomorph"]]
pairs_main = vbd_main.drop_duplicates(["base", "inflected"])
class_check = pd.DataFrame({"rows": vbd_main.past_class.value_counts(),
                            "pairs": pairs_main.past_class.value_counts(),
                            "pairs_in_classes_csv": vbd_classes.past_class.value_counts()})
print(class_check)
print(pairs_main[pairs_main.past_class == "regular"].allomorph.value_counts())
for c in ["regular", "semi_regular", "devoicing", "vowel_change", "suppletive"]:
    assert class_check.loc[c, "pairs"] > 0, f"no usable rows for class {c}"
assert (class_check.pairs == class_check.pairs_in_classes_csv.fillna(0)).all(), "class counts differ from vbd_classes.csv"
del vbd_main, pairs_main


In [ ]:
inflection_results_df = pd.read_parquet(inflection_results_path)

In [ ]:
most_common_allomorphs = pd.read_csv(most_common_allomorphs_path)
false_friends_df = pd.read_csv(false_friends_path)

## Homophone preparation

In [ ]:
from collections import defaultdict


pron2label = defaultdict(set)
for label, rows in cut_phonemic_forms.groupby("label"):
    for pron in set(rows):
        pron2label[pron].add(label)

homophone_map = defaultdict(set)
for label_idx, label in enumerate(state_space_spec.labels):
    for pron in set(cut_phonemic_forms.loc[label]):
        homophone_map[label] |= pron2label[pron]

In [ ]:
# Prepare to exclude predictions of homophones from analogy evaluations.
# create a map from label idx -> all label idxs which should be ignored.
homophone_map = {state_space_spec.labels.index(label): {state_space_spec.labels.index(hom) for hom in homs}
                 for label, homs in homophone_map.items()}

## Behavioral tests

In [ ]:
# study 3 most frequent allomorphs of each inflection
transfer_allomorphs = most_common_allomorphs.groupby("inflection").most_common_allomorph.apply(lambda xs: xs.value_counts().head(3).index.tolist()).to_dict()

In [ ]:
# experiment = "unambiguous-NNS_Z_to_NNS_Z"
# config = experiments[experiment]
# ret = analogy.run_experiment_equiv_level(
#     experiment, config, state_space_spec, all_cross_instances,
#     agg, agg_src,
#     num_samples=20,
#     device="cpu",
#     include_idxs_in_predictions=homophone_map,
#     verbose=True,
# )

In [ ]:
# Results are saved per experiment so that an interrupted run can resume; each experiment
# reseeds the RNG, so resumed results are identical to an uninterrupted run. Unlike run.ipynb, partial
# files are named by experiment, not by position, and experiments.json records the experiment set, so a
# changed experiment list can never load another experiment's results.
import json
import time
from pathlib import Path
partial_dir = Path(output_dir) / "partial_results"
partial_dir.mkdir(parents=True, exist_ok=True)
manifest = {"num_samples": num_samples, "seed": seed, "embeddings_path": embeddings_path, "experiments": experiments}
manifest_path = Path(output_dir) / "experiments.json"
if manifest_path.exists() and json.loads(manifest_path.read_text()) != manifest:
    raise RuntimeError(f"{output_dir} holds results of a different run; use a fresh output_dir")
manifest_path.write_text(json.dumps(manifest, indent=2))

experiment_results, timings = {}, []
for experiment, config in tqdm(experiments.items(), unit="experiment"):
    partial_path = partial_dir / f"{experiment}.csv"
    if partial_path.exists():
        experiment_results[experiment] = pd.read_csv(partial_path, index_col=0)
        continue
    t0 = time.time()
    result = analogy.run_experiment_equiv_level(
        experiment, config,
        state_space_spec, all_cross_instances,
        agg, agg_src,
        num_samples=num_samples,
        seed=seed,
        device="cuda")
    assert len(result) > 0, f"{experiment} returned no trials"
    result.to_csv(partial_path)
    experiment_results[experiment] = result
    timings.append({"experiment": experiment, "seconds": time.time() - t0, "n_trials": len(result)})

pd.DataFrame(timings).to_csv(f"{output_dir}/experiment_timings.csv", index=False)
experiment_results = pd.concat(experiment_results, names=["experiment"])
experiment_results["correct"] = experiment_results.predicted_label == experiment_results.gt_label
experiment_results.groupby("experiment", sort=False).agg(n_trials=("correct", "size"), top1=("correct", "mean"),
                                                          n_target_verbs=("base_to", "nunique"))


### Save

In [ ]:
experiment_results.to_csv(f"{output_dir}/experiment_results.csv")